
# CRISP-DM Step 3: Data Preparation — Hands-on Lab

**วัตถุประสงค์**
- เรียนรู้การเลือกข้อมูล (Select Data) ที่สอดคล้องกับเป้าหมายธุรกิจ
- ทำความสะอาดและแปลงข้อมูล (Clean & Transform): missing, outliers, types, join, encode
- สร้างฟีเจอร์ใหม่ (Feature Engineering): RFM, time-based, cyclical, ratios
- รวบรวมเป็น **Ready-for-Model Dataset** พร้อมนิยาม target (สังเคราะห์)



## 0) สร้างชุดข้อมูลจำลองสำหรับ Lab
ชุดข้อมูลทรานแซกชัน `df_txn` (2 ปี) พร้อมข้อมูลหมวดสินค้าและอายุลูกค้า เพื่อให้ทุกส่วนใน Data Preparation ทำงานได้ครบถ้วน


In [1]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(7)

# จำนวนลูกค้าและรายการ
n_customers = 5000
n_orders = 120_000

# ลูกค้า
cust_ids = [f"C{100000+i}" for i in range(n_customers)]
ages = np.clip(np.random.normal(35, 10, size=n_customers).round(), 15, 85)
# แทรก Missing อายุ ~10%
age_missing_mask = np.random.rand(n_customers) < 0.10
ages[age_missing_mask] = np.nan
df_cust = pd.DataFrame({'customer_id': cust_ids, 'customer_age': ages})

# หมวดสินค้า
categories = ['Electronics','Grocery','Fashion','Beauty','Home','Sports']
prod_ids = [f"P{1000+i}" for i in range(300)]
prod_cats = np.random.choice(categories, size=len(prod_ids), replace=True)
df_prod = pd.DataFrame({'product_id': prod_ids, 'product_category': prod_cats})

# สร้างออเดอร์ 2 ปี (2023-01-01 ถึง 2024-12-31)
dates = pd.date_range('2023-01-01', '2024-12-31', freq='D')
order_dates = np.random.choice(dates, size=n_orders, replace=True)

df_txn = pd.DataFrame({
    'order_id': np.arange(1, n_orders+1),
    'customer_id': np.random.choice(cust_ids, size=n_orders, replace=True),
    'product_id': np.random.choice(prod_ids, size=n_orders, replace=True),
    'order_date': pd.to_datetime(order_dates),
    'quantity': np.random.poisson(lam=2.5, size=n_orders).astype(int) + 1,
    'amount': np.round(np.random.gamma(shape=2.2, scale=180.0, size=n_orders), 2)
})

# ผสานข้อมูลอายุลูกค้าและหมวดหมู่สินค้า
df_txn = df_txn.merge(df_cust, on='customer_id', how='left')
df_txn = df_txn.merge(df_prod, on='product_id', how='left')

# แทรก Missing & Outliers
mask_amt = np.random.rand(n_orders) < 0.02
df_txn.loc[mask_amt, 'amount'] = np.nan

out_idx_q = np.random.choice(df_txn.index, size=800, replace=False)
df_txn.loc[out_idx_q, 'quantity'] = 1000 + np.random.randint(1, 50, size=800)

out_idx_a = np.random.choice(df_txn.index.difference(out_idx_q), size=800, replace=False)
df_txn.loc[out_idx_a, 'amount'] = 1_000_000 + np.random.randint(1, 50_000, size=800)

df_txn.sample(5, random_state=1)


ModuleNotFoundError: No module named 'matplotlib'


## 1) เลือกข้อมูลที่ต้องใช้ (Select Data)
- เลือกช่วงเวลาให้สอดคล้องกับโจทย์ (ตัวอย่าง: 12 เดือนล่าสุด)
- เลือก granularity ให้ตรงงาน (เช่น ลูกค้า-ต่อ-เดือน)
- หลีกเลี่ยง data leakage: ใช้เฉพาะข้อมูลที่รู้ได้ ณ เวลาทำนายจริง


In [ ]:

# 12 เดือนล่าสุด
cutoff_start = pd.Timestamp('2024-01-01')
df_last_12m = df_txn[df_txn['order_date'] >= cutoff_start].copy()

orders_per_month = (df_last_12m
                    .set_index('order_date')
                    .resample('M')['order_id']
                    .nunique())

print("ช่วงข้อมูล:", df_last_12m['order_date'].min(), "→", df_last_12m['order_date'].max())
print("แถว:", len(df_last_12m))
orders_per_month.head()



## 2) แปลง/รวม/ทำความสะอาด (Clean & Transform)
- Types, Missing, Outliers, One-Hot


In [ ]:

df = df_last_12m.copy()

# Types
df['order_date'] = pd.to_datetime(df['order_date'])
df['product_category'] = df['product_category'].astype('category')

# Missing
df['customer_age_imputed'] = df['customer_age'].fillna(df['customer_age'].median())
df['customer_age_is_missing'] = df['customer_age'].isna().astype(int)
df['amount_imputed'] = df['amount'].fillna(df['amount'].median())

# Outliers (clip p99)
q_qty = df['quantity'].quantile(0.99)
q_amt = df['amount_imputed'].quantile(0.99)
df['quantity_clip'] = df['quantity'].clip(upper=q_qty)
df['amount_clip'] = df['amount_imputed'].clip(upper=q_amt)

# One-hot
df_ohe = pd.get_dummies(df, columns=['product_category'], drop_first=True)
df_ohe.filter(regex='^(customer_age_imputed|customer_age_is_missing|quantity_clip|amount_clip|product_category_)').head()



## 3) Feature Engineering
- Avg orders/month ต่อ "ลูกค้า"
- RFM (recency, frequency, monetary)
- Cyclical time features
- Ratio features


In [ ]:

# Avg orders per month (customer-level)
m = (df.assign(month=lambda x: x['order_date'].dt.to_period('M').dt.to_timestamp())
       .groupby(['customer_id','month'])['order_id'].nunique()
       .reset_index(name='orders_per_month'))

avg_orders_per_month = (m.groupby('customer_id')['orders_per_month']
                          .mean()
                          .reset_index(name='avg_orders_per_month'))

# RFM
ref_date = df['order_date'].max() + pd.Timedelta(days=1)
rfm = (df.groupby('customer_id')
         .agg(last_purchase=('order_date','max'),
              frequency=('order_id','nunique'),
              monetary=('amount_imputed','sum'))
         .reset_index())
rfm['recency_days'] = (ref_date - rfm['last_purchase']).dt.days
rfm = rfm.drop(columns=['last_purchase'])

# Cyclical
df['month'] = df['order_date'].dt.month
df['month_sin'] = np.sin(2*np.pi*df['month']/12)
df['month_cos'] = np.cos(2*np.pi*df['month']/12)

# Ratio
rfm['avg_basket_value'] = (rfm['monetary'] / rfm['frequency']).replace(np.inf, np.nan)

# Merge features
feat_customer = rfm.merge(avg_orders_per_month, on='customer_id', how='left')
feat_customer.head()



## 4) Ready-for-Model Dataset
- นิยาม target สังเคราะห์: recency_days > 60 ⇒ churned=1
- เลือกฟีเจอร์ และเตรียม X, y


In [ ]:

feat = feat_customer.copy()
feat['churned'] = (feat['recency_days'] > 60).astype(int)

feature_cols = ['recency_days','frequency','monetary','avg_basket_value','avg_orders_per_month']
X = feat[feature_cols].copy()
y = feat['churned'].copy()

print("Ready-for-Model shape:", X.shape, y.shape)
feat.head()



## 5) (ออปชัน) Pipeline ตัวอย่างด้วย scikit-learn
โค้ดนี้จะรันก็ต่อเมื่อมี scikit-learn ในสภาพแวดล้อม ถ้าไม่มีจะถูกข้ามโดยอัตโนมัติ


In [ ]:

try:
    from sklearn.model_selection import train_test_split
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import Pipeline
    from sklearn.linear_model import LogisticRegression

    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

    pipe = Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000))
    ])

    pipe.fit(X_tr, y_tr)
    print("Train score:", pipe.score(X_tr, y_tr))
    print("Test  score:", pipe.score(X_te, y_te))
except Exception as e:
    print("ข้าม pipeline (ต้องใช้ scikit-learn):", e)



## 6) แบบฝึกหัด (TODO)
1. สร้างฟีเจอร์ time-based เพิ่มเติม เช่น `orders_30d`, `orders_90d` ต่อ **ลูกค้า**
2. ทดลองตัด outliers แบบ winsorize ที่ p1/p99 แล้วเทียบผลกับ clip เดิม
3. สร้าง target แบบ "ลูกค้าที่จะซื้ออีกภายใน 30 วันถัดไป" (ระวัง leakage) และลองเทียบสัดส่วน class
4. ลอง encoding หมวดหมู่รูปแบบอื่น (เช่น ordinal/leave-one-out) และประเมินผล
